In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
import datetime
import scipy.stats
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.colors as cl
import matplotlib.dates as mdates
import matplotlib
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS
import seaborn as sns
from scipy.optimize import curve_fit


In [ ]:
def line_fit(X,Y):
    n = len(X)
    P, residuals, rank, singular_values, rcond= np.polyfit(X,Y,1, full=True)
    dx = np.max(X) - np.min(X)
    X_new = np.array([np.min(X)-dx*0.2, np.max(X)+dx*0.2])
    Y_new = np.polyval(P, X_new)
    rmse = np.sqrt(residuals[0] / n)
    return X_new, Y_new, P, rmse

In [ ]:
data = pd.read_excel('CoralDepthData.xlsx')
data.rename(columns={data.columns[0]:'id', data.columns[1]:'d13c', data.columns[2]:'d18o'}, inplace=True)
data['path_name'] = [x[1] for x in data['id'].str.split(' ')]
data['path_num'] = [str.replace(x[-1],'A','') for x in data['id'].str.split(' ')]
depths = np.array([-105, -95, -68, -58, -48, -35, -15]) - 60
#depths = np.array([-105, -95, -73, -63, -53, -42, -32.5]) - 60
paths = np.unique(data['path_name'].values)
path_depth = dict(zip(paths,depths))
data['depth'] = data['path_name'].map(path_depth)
data

In [ ]:
paths = data['path_name'].unique().astype(str)

fig, ax = plt.subplots(1, figsize=(8,5))

for p in paths:

    data_iter = data[data['path_name'] == p]
    ax.plot(data_iter['path_num'] , data_iter['d13c'] ,".-", label = "Depth "+p)
ax.legend()
ax.grid()

ax.set_title(r"$\delta13$C")
fig.savefig("d13C.pdf")

In [ ]:
path_depth

In [ ]:
data

In [ ]:
depths

In [ ]:
path_depth['A']

In [ ]:
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

for i,p in enumerate(paths):
    
    data_iter = data[data['path_name'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'],data_iter['depth'], '.', color=c)
    ax.errorbar(x=data_iter['d13c'].mean(), 
                y=np.unique(data_iter['depth']), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)
x = data.groupby('path_name')['d13c'].mean().values
y = depths
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend()
ax.set_ylim(-180, -60)
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -60, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')

#fig.savefig('depth_d13c.pdf')

In [ ]:
viridis_cmap = plt.get_cmap('viridis')
colors = viridis_cmap(depths/depths.min()-0.1)

fig, ax = plt.subplots(1, figsize=(6, 6))

paths = ['A','C','D']
for i,p in enumerate(paths):
    
    data_iter = data[data['path_name'] == p]
    c = colors[i]
    ax.plot(data_iter['d13c'], data_iter['depth'], '.', color=c)
    ax.errorbar(x= data_iter['d13c'].mean(), 
                y= data_iter['depth'].unique(), yerr=0, fmt='xr',
                xerr=data_iter['d13c'].std(), ecolor=c,
                capsize=10)
                #label='mean')
    ax.text(-0.9, path_depth[p]-2, p, color=c)

data_iter = data[data['path_name'].isin(paths)]
x = data_iter.groupby('path_name')['d13c'].mean().values
y = data_iter['depth'].unique()
X, Y, P, rmse= line_fit(x,y)

ax.plot(X,Y, color='r', label=f'y = {P[0]:.3f}x+{P[1]:.3f}'+'\n'+f'RMSE={rmse:.3f}')

ax.legend()
ax.set_ylim(-180, -60)
ax.set_xlim(-3.5, -1)
ax.set_yticks(np.arange(-180, -60, 10))
ax.grid()
ax.set_ylabel('Water Depth (cm)')
ax.set_xlabel(r'$\delta^{13}C$ (‰)')
ax.set_title('Old Data [only A C D]')

fig.savefig('depth_d13c_old-ACD.pdf')

In [ ]:
paths = ['A', 'C', 'D']
data_iter = data[data['path_name'].isin(paths)]
data_iter

In [ ]:
data_iter['depth'].unique()

In [ ]:
data_iter['depth'].unique()